# Customer Support Chatbot Using Deep Learning and NLP
**Google Colab project notebook**

This notebook trains an intent-classification chatbot from the Bitext customer-service CSV files, evaluates it on the held-out test set, saves the model and metadata, and creates a Streamlit interface.

## Before you start
1. Run cells from top to bottom.
2. When prompted, upload the **training, validation, and testing CSV files**.
3. Training may take several minutes.
4. The chatbot predicts an intent and selects a predefined response template. It does not connect to real order, payment, refund, or account systems.

## 1. Install required libraries

In [ ]:
!pip -q install tensorflow pandas numpy scikit-learn matplotlib seaborn streamlit

## 2. Upload the three datasets

In [ ]:
from google.colab import files
uploaded = files.upload()
print("Uploaded files:")
for name in uploaded:
    print(" -", name)

## 3. Load the datasets and perform basic checks

In [ ]:
import os, json, random, glob, shutil, time, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

def find_uploaded_csv(keyword):
    matches = [
        f for f in os.listdir("/content")
        if f.lower().endswith(".csv") and keyword in f.lower()
    ]
    if not matches:
        raise FileNotFoundError(
            f"Could not find a CSV containing '{keyword}'. "
            "Please upload all three Bitext CSV files."
        )
    return os.path.join("/content", matches[0])

TRAIN_FILE = find_uploaded_csv("training")
VAL_FILE = find_uploaded_csv("validation")
TEST_FILE = find_uploaded_csv("testing")

train_df = pd.read_csv(TRAIN_FILE)
val_df = pd.read_csv(VAL_FILE)
test_df = pd.read_csv(TEST_FILE)

required_columns = {"utterance", "intent", "category", "tags"}
for name, df in [("Training", train_df), ("Validation", val_df), ("Testing", test_df)]:
    missing = required_columns - set(df.columns)
    if missing:
        raise ValueError(f"{name} dataset is missing columns: {sorted(missing)}")
    print(f"\n{name} shape: {df.shape}")
    display(df.head())
    print("Missing values:\n", df.isnull().sum())
    print("Exact duplicate rows:", df.duplicated().sum())
    print("Duplicate utterances:", df["utterance"].duplicated().sum())

## 4. Clean data and inspect intent distribution

In [ ]:
def clean_dataset(df):
    df = df.copy()
    df["utterance"] = df["utterance"].fillna("").astype(str).str.strip()
    df["intent"] = df["intent"].fillna("").astype(str).str.strip()
    df = df[(df["utterance"] != "") & (df["intent"] != "")]
    # Keep dataset splits intact; only remove exact duplicate rows within each split.
    df = df.drop_duplicates().reset_index(drop=True)
    return df

train_df = clean_dataset(train_df)
val_df = clean_dataset(val_df)
test_df = clean_dataset(test_df)

train_intents = set(train_df["intent"].unique())
val_intents = set(val_df["intent"].unique())
test_intents = set(test_df["intent"].unique())
if train_intents != val_intents or train_intents != test_intents:
    raise ValueError("Intent labels differ across splits. Check dataset labels before training.")

print("Cleaned sizes:", {"training": len(train_df), "validation": len(val_df), "testing": len(test_df)})
print("Number of intents:", train_df["intent"].nunique())
display(train_df["intent"].value_counts().rename("count").to_frame())

plt.figure(figsize=(14, 6))
train_df["intent"].value_counts().plot(kind="bar")
plt.title("Intent Distribution — Training Dataset")
plt.xlabel("Intent")
plt.ylabel("Number of messages")
plt.xticks(rotation=75, ha="right")
plt.tight_layout()
plt.show()

train_df["message_length"] = train_df["utterance"].str.len()
plt.figure(figsize=(8, 4))
sns.histplot(train_df["message_length"], bins=30, kde=True)
plt.title("Customer Message Length Distribution")
plt.xlabel("Message length (characters)")
plt.tight_layout()
plt.show()

## 5. Encode labels and build the neural network

In [ ]:
MAX_TOKENS = 12000
SEQUENCE_LENGTH = 40
EMBEDDING_DIM = 64
BATCH_SIZE = 32
EPOCHS = 20

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train_df["intent"])
y_val = label_encoder.transform(val_df["intent"])
y_test = label_encoder.transform(test_df["intent"])
intent_names = label_encoder.classes_.tolist()
num_classes = len(intent_names)

X_train = train_df["utterance"].astype(str).to_numpy()
X_val = val_df["utterance"].astype(str).to_numpy()
X_test = test_df["utterance"].astype(str).to_numpy()

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH,
    standardize="lower_and_strip_punctuation",
    name="text_vectorization"
)
# Adapt vocabulary on training text only to avoid information leakage.
vectorizer.adapt(tf.data.Dataset.from_tensor_slices(X_train).batch(256))

inputs = tf.keras.Input(shape=(), dtype=tf.string, name="customer_message")
x = vectorizer(inputs)
x = tf.keras.layers.Embedding(MAX_TOKENS, EMBEDDING_DIM, name="word_embedding")(x)
x = tf.keras.layers.GlobalAveragePooling1D(name="global_average_pooling")(x)
x = tf.keras.layers.Dense(64, activation="relu", name="hidden_dense")(x)
x = tf.keras.layers.Dropout(0.30, name="dropout")(x)
outputs = tf.keras.layers.Dense(num_classes, activation="softmax", name="intent_probabilities")(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs, name="customer_support_intent_classifier")
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

## 6. Train the model

In [ ]:
os.makedirs("/content/outputs", exist_ok=True)
MODEL_PATH = "/content/customer_support_intent_model.keras"

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=3, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        MODEL_PATH, monitor="val_loss", save_best_only=True, verbose=1
    )
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

## 7. Plot training and validation accuracy/loss

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.title("Training and Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()

## 8. Evaluate on the held-out test dataset

In [ ]:
model = tf.keras.models.load_model(MODEL_PATH)
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
test_probabilities = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
y_pred = np.argmax(test_probabilities, axis=1)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f} ({test_accuracy * 100:.2f}%)")
report_text = classification_report(
    y_test, y_pred, labels=np.arange(num_classes),
    target_names=intent_names, zero_division=0
)
print("\nCLASSIFICATION REPORT\n")
print(report_text)

with open("/content/outputs/classification_report.txt", "w", encoding="utf-8") as f:
    f.write(report_text)

cm = confusion_matrix(y_test, y_pred, labels=np.arange(num_classes))
plt.figure(figsize=(14, 12))
sns.heatmap(cm, cmap="Blues", xticklabels=intent_names, yticklabels=intent_names)
plt.title("Confusion Matrix — Test Dataset")
plt.xlabel("Predicted intent")
plt.ylabel("Actual intent")
plt.xticks(rotation=90, fontsize=7)
plt.yticks(rotation=0, fontsize=7)
plt.tight_layout()
plt.show()

error_df = pd.DataFrame({
    "utterance": X_test,
    "actual_intent": [intent_names[i] for i in y_test],
    "predicted_intent": [intent_names[i] for i in y_pred],
    "confidence": np.max(test_probabilities, axis=1)
})
misclassified = error_df[error_df["actual_intent"] != error_df["predicted_intent"]]
display(misclassified.head(20))
misclassified.to_csv("/content/outputs/misclassified_examples.csv", index=False)

## 9. Create response templates and save the model metadata

In [ ]:
RESPONSE_TEMPLATES = {
    "cancel_order": "I can guide you on order cancellation. Please check your order details and the seller's cancellation policy.",
    "change_order": "Please review your order status to see whether changes are still available.",
    "change_shipping_address": "Please check your order status and account options as soon as possible to see whether the shipping address can be changed.",
    "check_cancellation_fee": "Cancellation fees depend on the applicable order terms. Please review the fee information for your order.",
    "check_invoice": "You can usually find invoice details in the order or billing section of your account.",
    "get_invoice": "Please check your order history or billing section for invoice download options.",
    "payment_issue": "I can help with general payment troubleshooting. Check the payment status and never share full card details or passwords.",
    "check_payment_methods": "Available payment methods are normally shown during checkout or in payment settings.",
    "get_refund": "For a refund request, review the seller's refund process and the status of your order.",
    "track_refund": "Refund timing depends on the seller and payment provider. Check your account or contact official support for an update.",
    "check_refund_policy": "Please review the seller's refund policy for eligibility, timelines, and conditions.",
    "track_order": "You can usually track an order from your order history or the tracking link in your confirmation message.",
    "delivery_period": "Delivery estimates depend on the destination, shipping method, and seller. Check the estimated date in your order details.",
    "delivery_options": "Available delivery options are generally shown during checkout.",
    "place_order": "To place an order, add the desired items to your cart and follow the checkout steps.",
    "set_up_shipping_address": "You can usually add a shipping address in account settings or during checkout.",
    "create_account": "Use the sign-up option and follow the instructions to create an account.",
    "edit_account": "Account details can generally be updated from your profile or account settings.",
    "delete_account": "Please review account settings or contact the service's official support team to request account deletion.",
    "recover_password": "Use the 'Forgot password' option and follow the password-reset instructions.",
    "registration_problems": "Check that required fields are complete and your email address is entered correctly. If the issue continues, contact support.",
    "switch_account": "If the service supports multiple accounts, sign out and sign in with the intended account.",
    "newsletter_subscription": "Newsletter subscription settings are usually available in your profile or at the bottom of newsletter emails.",
    "review": "You can usually submit feedback or a review from the product or order page.",
    "complaint": "I'm sorry you're experiencing a problem. Please describe the issue without including sensitive personal or payment information.",
    "contact_customer_service": "Please use the service's official customer-support channel for help with your specific case.",
    "contact_human_agent": "For help from a person, please use the service's official contact or live-chat option."
}

FALLBACK_RESPONSE = "I'm not fully sure I understood that. Could you rephrase your question or contact the service's official support team?"
CONFIDENCE_THRESHOLD = 0.60

for intent in intent_names:
    RESPONSE_TEMPLATES.setdefault(
        intent,
        "I can provide general guidance, but please contact official customer support for help with this specific request."
    )

metadata = {
    "project_title": "Customer Support Chatbot Using Deep Learning and NLP",
    "intent_names": intent_names,
    "response_templates": RESPONSE_TEMPLATES,
    "fallback_response": FALLBACK_RESPONSE,
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "max_tokens": MAX_TOKENS,
    "sequence_length": SEQUENCE_LENGTH,
    "embedding_dim": EMBEDDING_DIM,
    "seed": SEED
}

METADATA_PATH = "/content/customer_support_metadata.json"
with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

metrics = {
    "test_accuracy": float(test_accuracy),
    "test_loss": float(test_loss),
    "training_rows": len(train_df),
    "validation_rows": len(val_df),
    "testing_rows": len(test_df),
    "number_of_intents": len(intent_names),
    "epochs_completed": len(history.history["loss"])
}
with open("/content/outputs/evaluation_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print("Saved model:", MODEL_PATH)
print("Saved metadata:", METADATA_PATH)
print("Evaluation files are in /content/outputs/")
print(json.dumps(metrics, indent=2))

## 10. Test the chatbot inside Colab

In [ ]:
def predict_customer_message(message):
    probabilities = model.predict(np.array([str(message)], dtype=object), verbose=0)[0]
    predicted_id = int(np.argmax(probabilities))
    intent = intent_names[predicted_id]
    confidence = float(probabilities[predicted_id])

    if confidence < CONFIDENCE_THRESHOLD:
        response = FALLBACK_RESPONSE
    else:
        response = RESPONSE_TEMPLATES.get(intent, FALLBACK_RESPONSE)

    return {"intent": intent, "confidence": confidence, "response": response}

test_messages = [
    "Hello, can you help me?",
    "I want to cancel my order",
    "I need a refund",
    "Where is my package?"
]

for message in test_messages:
    result = predict_customer_message(message)
    print("\nCustomer:", message)
    print("Predicted intent:", result["intent"])
    print("Confidence:", f"{result['confidence']:.2%}")
    print("Chatbot:", result["response"])

## 11. Create the Streamlit app file

In [ ]:
%%writefile /content/app.py
import json
import numpy as np
import streamlit as st
import tensorflow as tf

MODEL_PATH = "/content/customer_support_intent_model.keras"
METADATA_PATH = "/content/customer_support_metadata.json"

st.set_page_config(page_title="Customer Support Chatbot", page_icon="💬", layout="centered")

@st.cache_resource
def load_artifacts():
    model = tf.keras.models.load_model(MODEL_PATH)
    with open(METADATA_PATH, "r", encoding="utf-8") as f:
        metadata = json.load(f)
    return model, metadata

model, metadata = load_artifacts()
st.title("💬 Customer Support Chatbot")
st.caption("Deep Learning + Natural Language Processing (NLP)")
st.write("Ask a question about orders, delivery, refunds, payments, accounts, or customer support.")
st.info("Academic prototype only: this app does not access real orders, payments, refunds, or customer accounts.")

if "messages" not in st.session_state:
    st.session_state.messages = [{
        "role": "assistant",
        "content": "Hello! I'm your customer support assistant. How can I help you today?"
    }]

for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])
        if message.get("details"):
            with st.expander("Prediction details"):
                st.write(message["details"])

prompt = st.chat_input("Type your customer-support question...")
if prompt:
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    probs = model.predict(np.array([prompt], dtype=object), verbose=0)[0]
    idx = int(np.argmax(probs))
    intent = metadata["intent_names"][idx]
    confidence = float(probs[idx])
    threshold = float(metadata.get("confidence_threshold", 0.60))

    if confidence < threshold:
        response = metadata["fallback_response"]
    else:
        response = metadata["response_templates"].get(intent, metadata["fallback_response"])

    details = {
        "Predicted intent": intent,
        "Confidence": f"{confidence:.1%}",
        "Fallback used": confidence < threshold
    }

    with st.chat_message("assistant"):
        st.markdown(response)
        with st.expander("Prediction details"):
            st.write(details)

    st.session_state.messages.append({
        "role": "assistant",
        "content": response,
        "details": details
    })

with st.sidebar:
    st.header("About this project")
    st.write("A neural-network text classifier predicts the intent of a customer message.")
    st.write(f"Supported intents: {len(metadata['intent_names'])}")
    st.write(f"Confidence threshold: {metadata.get('confidence_threshold', 0.60):.0%}")
    if st.button("Clear conversation"):
        st.session_state.messages = [{
            "role": "assistant",
            "content": "Hello! I'm your customer support assistant. How can I help you today?"
        }]
        st.rerun()

## 12. Launch Streamlit from Colab

**Run this cell to start the Streamlit server.** It starts a localtunnel URL so you can open the chat interface in your browser. Keep the tunnel cell running while demonstrating. Public tunnel links are temporary and may stop when Colab disconnects.

In [ ]:
# Start Streamlit in the background
!pkill -f "streamlit run /content/app.py" || true

In [ ]:
import subprocess, time, os
streamlit_log = open("/content/streamlit.log", "w")
streamlit_process = subprocess.Popen(
    ["streamlit", "run", "/content/app.py",
     "--server.port", "8501",
     "--server.address", "0.0.0.0",
     "--server.headless", "true"],
    stdout=streamlit_log,
    stderr=subprocess.STDOUT
)
time.sleep(5)
print("Streamlit started. Process ID:", streamlit_process.pid)
with open("/content/streamlit.log", "r") as f:
    print(f.read()[-2000:])

In [ ]:
!npm -q install -g localtunnel

In [ ]:
# Keep this cell running. Open the public URL printed by localtunnel.
!lt --port 8501

## 13. Download your project results

Download the model, metadata, evaluation metrics, and report outputs so you can keep them for your submission.

Run this cell when you are ready:

In [ ]:
from google.colab import files
files.download("/content/customer_support_intent_model.keras")
files.download("/content/customer_support_metadata.json")
files.download("/content/outputs/evaluation_metrics.json")
files.download("/content/outputs/classification_report.txt")

## Submission checklist
- [ ] Run every cell successfully.
- [ ] Save the actual test accuracy and classification report.
- [ ] Capture screenshots of the training curves, confusion matrix, and live chatbot.
- [ ] Explain that this is an intent-classification prototype with template-based responses.
- [ ] Do not claim it processes real orders, payments, or refunds.